# Lesson 08 Lab — Quantization Math: Scale, Zero Point, Group Size, and Error

**Puzzle:** Why does changing group size alter both model size and reconstruction error?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

The label INT4 hides the parameters that determine what four bits mean. Scale chooses the real interval covered by the codes, zero point chooses where real zero lands, and group size chooses how many values share one range estimate. Those choices change both reconstruction error and metadata, even before a deployment kernel enters the picture.


## 0. Predict before running

1. Derive symmetric INT4 quantize and dequantize equations for code range [-8, 7].
2. Predict how RMSE, scale count, and effective bits per weight change as group size shrinks.
3. Explain why saturation fraction alone does not rank quantizers.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Uniform quantization stores integer codes plus scale metadata and, for asymmetric schemes, zero points. Granularity may be per tensor, row/channel, or group/block.

- Scale maps a floating-point interval to a finite code range.
- Symmetric quantization fixes zero point at zero; asymmetric quantization can spend codes more efficiently on shifted data.
- Smaller groups adapt to local ranges but require more scale metadata.


## 2. Derive the mechanism

A common mapping is `q = clamp(round(x/s)+z, qmin, qmax)` and `x_hat = s(q-z)`. Symmetric INT4 typically uses `z=0` and a signed range near `[-8,7]`. Smaller groups estimate local ranges and reduce outlier sharing.

For symmetric signed b-bit quantization, let `qmax = 2^(b-1)-1`, `s = max(|x|)/qmax`, `q = clamp(round(x/s), -qmax-1, qmax)`, and `x̂ = s·q`. With asymmetric quantization a zero point z shifts the code grid: `q = clamp(round(x/s)+z, qmin, qmax)` and `x̂=s(q-z)`. Grouping repeats this calculation over local slices rather than the whole tensor.

If each group stores one FP16 scale, its metadata cost is `16/group_size` bits per weight. Nominal INT4 therefore becomes 5.0 effective bits at group size 16, 4.25 at 64, and 4.125 at 128 before padding or zero-point metadata. Smaller groups can isolate outliers but may be incompatible with the fastest backend kernels.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "08-quantization-math"
device = require_cuda()
torch.manual_seed(2026 + 8)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | one fixed outlier-containing 1024×1024 weight matrix |
| Candidate | symmetric INT4 with group sizes 16, 64, and 128 |
| Held constant | same codes, scale dtype assumption, grouping axis, seed, and error reference |
| Measurements | RMSE/cosine error, saturation fraction, scale count, effective bits per weight |
| Evidence | `numerical-model` |

**Experiment:** Quantize an outlier-containing matrix with INT4 group sizes 16, 64, and 128 and compare error plus metadata overhead.


## 5. Read the experiment code

The notebook holds the weight matrix fixed, changes only group size, and records both error and effective bits per weight.

The notebook holds the matrix and quantization formula fixed and changes only group size. Each candidate is dequantized back to floating point before error is measured. Metadata is computed from the number of scales, making the storage comparison honest instead of repeating the nominal four-bit label.

This is a numerical model. It does not pack nibbles, instantiate a production quantized linear layer, or time an INT4 kernel. That separation lets the lab answer the math question without overstating backend performance.

Only after these variables match the protocol should the cell be executed.


In [2]:
w = torch.randn(1024, 1024, device=device); w[:, ::97] *= 12
rows = []
for group in (16, 64, 128):
    q, scales, dq = symmetric_quantize(w, bits=4, group_size=group)
    metadata_bits = scales.numel() * 16
    rows.append({"group_size": group, "error": error_metrics(w, dq), "scale_count": scales.numel(),
                 "effective_bits_per_weight": round(4 + metadata_bits / w.numel(), 5),
                 "saturation_fraction": round((q.abs() == 7).float().mean().item(), 6)})
result = base_result(8, "numerical-model"); result.update({"shape": list(w.shape), "group_results": rows,
    "conclusion": "Smaller groups reduced local range sharing at the cost of more scale metadata."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Group 16 RMSE | 0.200316 |
| Group 16 effective bits | 5.000 bits/weight |
| Group 64 RMSE | 0.384361 |
| Group 64 effective bits | 4.250 bits/weight |
| Group 128 RMSE | 0.508112 |
| Group 128 effective bits | 4.125 bits/weight |


## 7. Interpret rather than merely print

Group size 16 produced the lowest RMSE, 0.200316, and cosine 0.992188, but required 65,536 scales and 5.0 effective bits per weight. At group size 128, scale count fell to 8,192 and effective storage to 4.125 bits, while RMSE rose to 0.508112 and cosine fell to 0.950873. Group size 64 sat between them.

The saturation fraction decreased with larger groups because the shared maximum widened each step size; fewer values landed on the extreme code, but reconstruction became coarser. This is why a lower saturation count is not automatically a better quantizer.

**Inspection rule:** Check saturation, error, and effective bits per value. Do not report the nominal four bits without scale overhead.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA numerical experiment isolates an algorithmic mechanism. It is not the paper's complete implementation and does not establish a production kernel speedup.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Smaller groups reduced local range sharing at the cost of more scale metadata.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "numerical-model",
  "executed_at_utc": "2026-08-07T14:45:26+00:00",
  "group_results": [
    {
      "effective_bits_per_weight": 5.0,
      "error": {
        "cosine": 0.99218798,
        "mae": 0.11072622,
        "max_abs": 2.84089684,
        "rmse": 0.20031616
      },
      "group_size": 16,
      "saturation_fraction": 0.077793,
      "scale_count": 65536
    },
    {
      "effective_bits_per_weight": 4.25,
      "error": {
        "cosine": 0.97160149,
        "mae": 0.25374436,
        "max_abs": 3.00970507,
        "rmse": 0.38436082
      },
      "group_size": 64,
      "saturation_fraction": 0.019199,
      "scale_count": 16384
    },
    {
      "effecti

## 9. Make the bounded decision

> Group size is an error–metadata–kernel compatibility decision, not a cosmetic configuration value.

**Acceptance/rollback:** Report nominal bits, scale/zero-point overhead, clipping rate, reconstruction error, group axis, and kernel-compatible group size together.

**Failure analysis:** Comparing only weight RMSE ignores how inputs weight different columns. Comparing only effective bits ignores alignment, padding, and scale loads. Finally, a group size with good numerical behavior can lose in production if the backend does not provide a fused kernel for that layout.


## 10. Extend the evidence

Add asymmetric zero points for shifted distributions, compare per-row and per-column grouping, and weight the error by held-out activations. Then pack two INT4 codes per byte and time a compatible native kernel so numerical, storage, and operator gates are all represented.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
